<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/fedavg_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 1: FedAvg (McMahan et al. 2017)
 Emaduddin Asdaq Syed Mohammed | CSC8639 MSc Data Science and AI  

---
**This notebook runs FedAvg only on the full dataset.**  
Run q-FedAvg and GIFAIR-FL in separate sessions simultaneously.

## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 34.2 MB/s eta 0:00:00
✓ Libraries installed


In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.0
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, parameters_to_ndarrays,
                          ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr     : {fl.__version__}")
print(f"torch    : {torch.__version__}")
print(f"numpy    : {np.__version__}")
print(f"GPU      : {torch.cuda.is_available()}")
print("✓ All libraries ready")

flwr     : 1.32.0
torch    : 2.11.0+cu128
numpy    : 2.0.2
GPU      : True
✓ All libraries ready


In [ ]:
from google.colab import drive
import time, os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=== Session Initialisation ===")
print(f"Device     : {device}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory     : {mem:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
    print("\n✓ GPU ready — safe to run training cells")
else:
    print("\n⚠ No GPU detected")
    print("  Runtime → Change runtime type → A100 GPU")
    print("  Then Runtime → Restart session → run this cell again")

=== Session Initialisation ===
Device     : cuda
GPU        : NVIDIA L4
Memory     : 23.7 GB
CUDA       : 12.8

✓ GPU ready — safe to run training cells


## Section 2 — Dataset Download and Image Indexing

In [ ]:
import os, time, zipfile

ZIP_PATH     = '/content/drive/MyDrive/dissertation/archive.zip'
EXTRACT      = '/content/nih_unzipped'
DATASET_PATH = EXTRACT

os.makedirs(EXTRACT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    total     = len(all_files)

print(f"ZIP contains : {total:,} files")
print(f"Extracting to: {EXTRACT}\n")

t0 = time.time()

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for i, file in enumerate(all_files):
        z.extract(file, EXTRACT)
        pct  = (i + 1) / total * 100
        bar  = '█' * int(pct / 2) + '││' * (50 - int(pct / 2))
        mins = (time.time() - t0) / 60
        print(f"\r[{bar}] {pct:5.1f}% | {i+1:,}/{total:,} | {mins:.1f} min",
              end='', flush=True)

elapsed = time.time() - t0
print(f"\n\n✓ Extraction complete in {elapsed/60:.1f} minutes")

ZIP contains : 112,128 files
Extracting to: /content/nih_unzipped

[██████████████████████████████████████████████████] 100.0% | 112,128/112,128 | 15.1 min

✓ Extraction complete in 15.1 minutes


In [ ]:
DATASET_PATH = '/content/nih_unzipped'

print("=== NIH ChestX-ray14 — Top Level Contents ===\n")
print(f"{'Name':<35} {'Type':<8} {'Size/Items':>12}")
print("-" * 58)
for item in sorted(os.listdir(DATASET_PATH)):
    p = os.path.join(DATASET_PATH, item)
    if os.path.isdir(p):
        print(f"{item:<35} {'DIR':<8} {len(os.listdir(p)):>10} items")
    else:
        print(f"{item:<35} {'FILE':<8} {os.path.getsize(p)/1e6:>9.1f} MB")

=== NIH ChestX-ray14 — Top Level Contents ===

Name                                Type       Size/Items
----------------------------------------------------------
ARXIV_V5_CHESTXRAY.pdf              FILE           9.0 MB
BBox_List_2017.csv                  FILE           0.1 MB
Data_Entry_2017.csv                 FILE           7.9 MB
FAQ_CHESTXRAY.pdf                   FILE           0.1 MB
LOG_CHESTXRAY.pdf                   FILE           0.0 MB
README_CHESTXRAY.pdf                FILE           0.8 MB
images_001                          DIR               1 items
images_002                          DIR               1 items
images_003                          DIR               1 items
images_004                          DIR               1 items
images_005                          DIR               1 items
images_006                          DIR               1 items
images_007                          DIR               1 items
images_008                          DIR               

In [ ]:
print("Building image path index...")
image_path_dict = {}
for folder in sorted(os.listdir(DATASET_PATH)):
    if folder.startswith('images_'):
        img_dir = os.path.join(DATASET_PATH, folder, 'images')
        if os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.endswith('.png'):
                    image_path_dict[f] = os.path.join(img_dir, f)
print(f"✓ Indexed {len(image_path_dict):,} images")
sample = '00000001_000.png'
if sample in image_path_dict:
    print(f"✓ Spot check OK : {image_path_dict[sample]}")

Building image path index...
✓ Indexed 112,120 images
✓ Spot check OK : /content/nih_unzipped/images_001/images/00000001_000.png


## Section 3 — Metadata Loading, Exploration and Cleaning

In [ ]:
import pandas as pd

df_raw = pd.read_csv(f"{DATASET_PATH}/Data_Entry_2017.csv")
print(f"Rows    : {len(df_raw):,}")
print(f"Columns : {len(df_raw.columns)}")
print("\nColumn names:")
for i, col in enumerate(df_raw.columns):
    print(f"  {i+1:>2}. {col}")

Rows    : 112,120
Columns : 12

Column names:
   1. Image Index
   2. Finding Labels
   3. Follow-up #
   4. Patient ID
   5. Patient Age
   6. Patient Gender
   7. View Position
   8. OriginalImage[Width
   9. Height]
  10. OriginalImagePixelSpacing[x
  11. y]
  12. Unnamed: 11


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("=== Data Types ===")
print(df_raw.dtypes)
print("\n=== First 3 Rows ===")
print(df_raw.head(3).to_string())

=== Data Types ===
Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
Unnamed: 11                    float64
dtype: object

=== First 3 Rows ===
        Image Index          Finding Labels  Follow-up #  Patient ID  Patient Age Patient Gender View Position  OriginalImage[Width  Height]  OriginalImagePixelSpacing[x     y]  Unnamed: 11
0  00000001_000.png            Cardiomegaly            0           1           58              M            PA                 2682     2749                        0.143  0.143          NaN
1  00000001_001.png  Cardiomegaly|Emphysema            1           1           58              M 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("=== Missing Values ===\n")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    status = '✓ none' if n == 0 else f'⚠  {n:,} missing'
    print(f"  {col:<35} {status}")

=== Missing Values ===

  Image Index                         ✓ none
  Finding Labels                      ✓ none
  Follow-up #                         ✓ none
  Patient ID                          ✓ none
  Patient Age                         ✓ none
  Patient Gender                      ✓ none
  View Position                       ✓ none
  OriginalImage[Width                 ✓ none
  Height]                             ✓ none
  OriginalImagePixelSpacing[x         ✓ none
  y]                                  ✓ none
  Unnamed: 11                         ⚠  112,120 missing


In [ ]:
all_labels   = df_raw['Finding Labels'].str.split('|').explode()
label_counts = all_labels.value_counts()
print(f"{'Label':<25} {'Count':>8} {'%':>7}")
print("-" * 42)
for label, count in label_counts.items():
    print(f"{label:<25} {count:>8,} {count/len(df_raw)*100:>6.1f}%")
print(f"\nNo Finding rate : {label_counts['No Finding']/len(df_raw)*100:.1f}%")

Label                        Count       %
------------------------------------------
No Finding                  60,361   53.8%
Infiltration                19,894   17.7%
Effusion                    13,317   11.9%
Atelectasis                 11,559   10.3%
Nodule                       6,331    5.6%
Mass                         5,782    5.2%
Pneumothorax                 5,302    4.7%
Consolidation                4,667    4.2%
Pleural_Thickening           3,385    3.0%
Cardiomegaly                 2,776    2.5%
Emphysema                    2,516    2.2%
Edema                        2,303    2.1%
Fibrosis                     1,686    1.5%
Pneumonia                    1,431    1.3%
Hernia                         227    0.2%

No Finding rate : 53.8%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
sex = df_raw['Patient Gender'].value_counts()
pct = df_raw['Patient Gender'].value_counts(normalize=True).mul(100)
print(f"{'Sex':<10} {'Count':>8} {'%':>8}")
print("-" * 28)
for s in sex.index:
    print(f"{s:<10} {sex[s]:>8,} {pct[s]:>7.2f}%")

Sex           Count        %
----------------------------
M            63,340   56.49%
F            48,780   43.51%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import numpy as np

print(f"Min age   : {df_raw['Patient Age'].min()}")
print(f"Max age   : {df_raw['Patient Age'].max()}")
print(f"Mean age  : {df_raw['Patient Age'].mean():.1f}")
print(f"Median age: {df_raw['Patient Age'].median():.0f}")
print()
above_100 = df_raw[df_raw['Patient Age'] > 100]
print(f"Rows with age > 100 : {len(above_100)}")
print("Values above 100 (data entry errors):")
print(above_100['Patient Age'].value_counts().sort_index())
print("\nConclusion: all ages > 100 are data entry errors — will be removed")

Min age   : 1
Max age   : 414
Mean age  : 46.9
Median age: 49

Rows with age > 100 : 16
Values above 100 (data entry errors):
Patient Age
148    2
149    1
150    1
151    1
152    1
153    1
154    1
155    2
411    1
412    3
413    1
414    1
Name: count, dtype: int64

Conclusion: all ages > 100 are data entry errors — will be removed


In [ ]:
folder_001   = os.path.join(DATASET_PATH, 'images_001', 'images')
sample_files = sorted(os.listdir(folder_001))[:5]
print(f"{'Filename':<25} {'In Metadata':>12} {'Finding Labels'}")
print("-" * 65)
for fname in sample_files:
    match = df_raw[df_raw['Image Index'] == fname]
    if len(match) > 0:
        print(f"{fname:<25} {'✓ YES':>12}  {match.iloc[0]['Finding Labels']}")
    else:
        print(f"{fname:<25} {'⚠ NOT FOUND':>12}")

Filename                   In Metadata Finding Labels
-----------------------------------------------------------------
00000001_000.png                 ✓ YES  Cardiomegaly
00000001_001.png                 ✓ YES  Cardiomegaly|Emphysema
00000001_002.png                 ✓ YES  Cardiomegaly|Effusion
00000002_000.png                 ✓ YES  No Finding
00000003_000.png                 ✓ YES  Hernia


In [ ]:
df = df_raw.copy()

df = df.rename(columns={'Patient Gender': 'Patient Sex'})
print("✓ Step 1 — Patient Gender → Patient Sex")

null_cols = [c for c in df.columns if df[c].isnull().all()]
df        = df.drop(columns=null_cols)
print(f"✓ Step 2 — Removed null columns: {null_cols}")

before = len(df)
df     = df[df['Patient Age'] <= 100]
print(f"✓ Step 3 — Removed {before - len(df)} rows (age > 100 — data entry errors)")
print(f"           Age range now: {df['Patient Age'].min()} – {df['Patient Age'].max()}")

df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)
print(f"✓ Step 4 — Binary label added | Pathology rate: {df['label'].mean()*100:.1f}%")

df['Age Group'] = pd.cut(
    df['Patient Age'],
    bins   = [0, 20, 40, 60, 80, 101],
    labels = ['0-20', '20-40', '40-60', '60-80', '80+']
)
print("✓ Step 5 — Age groups: 0-20 | 20-40 | 40-60 | 60-80 | 80+")

df['image_path'] = df['Image Index'].map(image_path_dict)
matched   = df['image_path'].notna().sum()
unmatched = df['image_path'].isna().sum()
print(f"✓ Step 6 — Image paths linked: {matched:,} matched | {unmatched} unmatched")
print(f"\nFinal dataset  : {len(df):,} rows | {len(df.columns)} columns")

✓ Step 1 — Patient Gender → Patient Sex
✓ Step 2 — Removed null columns: ['Unnamed: 11']
✓ Step 3 — Removed 16 rows (age > 100 — data entry errors)
           Age range now: 1 – 95
✓ Step 4 — Binary label added | Pathology rate: 46.2%
✓ Step 5 — Age groups: 0-20 | 20-40 | 40-60 | 60-80 | 80+


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ Step 6 — Image paths linked: 112,104 matched | 0 unmatched

Final dataset  : 112,104 rows | 14 columns


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

In [ ]:
import flwr as fl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
from typing import Dict, List, Optional, Tuple
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
import numpy as np
import warnings, json, os, time, copy
warnings.filterwarnings('ignore')

print(f"✓ Flower   : {fl.__version__}")
print(f"✓ PyTorch  : {torch.__version__}")
print(f"✓ GPU      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU name : {torch.cuda.get_device_name(0)}")

✓ Flower   : 1.32.0
✓ PyTorch  : 2.11.0+cu128
✓ GPU      : True
✓ GPU name : NVIDIA L4


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\n✓ GPU confirmed — safe to proceed")
else:
    print("\n⚠ No GPU — go to Runtime → Change runtime type")

Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB

✓ GPU confirmed — safe to proceed


In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")
print(f"✓ cuDNN bench : enabled")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2
✓ cuDNN bench : enabled


## Section 5 — Dirichlet Partition → 5 Hospital Clients

In [ ]:
np.random.seed(42)

NUM_CLIENTS    = 5
ALPHA          = 0.5
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

print(f"Clients  : {NUM_CLIENTS}")
print(f"Alpha    : {ALPHA}  — Dir(α=0.5) [Morafah et al. 2022]")
print(f"Seed     : 42")
print(f"Clients  : {HOSPITAL_NAMES}")

Clients  : 5
Alpha    : 0.5  — Dir(α=0.5) [Morafah et al. 2022]
Seed     : 42
Clients  : ['Hospital_A', 'Hospital_B', 'Hospital_C', 'Hospital_D', 'Hospital_E']


In [ ]:
idx_0 = np.where(df['label'].values == 0)[0]
idx_1 = np.where(df['label'].values == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

print(f"Class 0 — No Finding : {len(idx_0):,} ({len(idx_0)/len(df)*100:.1f}%)")
print(f"Class 1 — Pathology  : {len(idx_1):,} ({len(idx_1)/len(df)*100:.1f}%)")
print(f"Total                : {len(df):,}")

Class 0 — No Finding : 60,353 (53.8%)
Class 1 — Pathology  : 51,751 (46.2%)
Total                : 112,104


In [ ]:
props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

print(f"{'Client':<14} {'No Finding%':>12} {'Pathology%':>12}")
print("-" * 40)
for i, name in enumerate(HOSPITAL_NAMES):
    print(f"{name:<14} {props_0[i]*100:>11.1f}% {props_1[i]*100:>11.1f}%")

Client          No Finding%   Pathology%
----------------------------------------
Hospital_A            40.8%        71.2%
Hospital_B             0.0%        20.5%
Hospital_C            53.1%         5.6%
Hospital_D             0.8%         2.0%
Hospital_E             5.2%         0.7%


In [ ]:
splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(HOSPITAL_NAMES):
    idx = np.concatenate([
        idx_0[ptr0 : ptr0 + splits_0[i]],
        idx_1[ptr1 : ptr1 + splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy().reset_index(drop=True)
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

assigned = sum(len(d) for d in clients.values())
print(f"Total assigned : {assigned:,}")
print(f"Total dataset  : {len(df):,}")
print(f"Match          : {'✓' if assigned == len(df) else '⚠ MISMATCH'}")

Total assigned : 112,104
Total dataset  : 112,104
Match          : ✓


In [ ]:
print("=" * 80)
print(f"HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α={ALPHA}) [Morafah et al. 2022]")
print("=" * 80)
print(f"\n{'Client':<14} {'Images':>8} {'Pathology%':>12} {'No Finding%':>13} "
      f"{'Age Mean':>9} {'Male%':>7}")
print("-" * 68)

for name, data in clients.items():
    n      = len(data)
    path   = data['label'].mean() * 100
    nof    = 100 - path
    age    = data['Patient Age'].mean()
    male   = (data['Patient Sex'] == 'M').mean() * 100
    print(f"{name:<14} {n:>8,} {path:>11.1f}% {nof:>12.1f}% {age:>8.1f} {male:>6.1f}%")

rates    = [d['label'].mean() for d in clients.values()]
variance = np.var(rates)
print("-" * 68)
print(f"\nPathology range : {min(rates)*100:.1f}% – {max(rates)*100:.1f}%")
print(f"Variance        : {variance:.6f}")
print(f"\n✓ Non-IID confirmed")

HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α=0.5) [Morafah et al. 2022]

Client           Images   Pathology%   No Finding%  Age Mean   Male%
--------------------------------------------------------------------
Hospital_A       61,505        59.9%         40.1%     47.2   56.4%
Hospital_B       10,655        99.7%          0.3%     48.2   56.8%
Hospital_C       34,936         8.2%         91.8%     45.9   56.6%
Hospital_D        1,495        69.1%         30.9%     47.8   58.0%
Hospital_E        3,513        10.0%         90.0%     46.3   55.6%
--------------------------------------------------------------------

Pathology range : 8.2% – 99.7%
Variance        : 0.125535

✓ Non-IID confirmed


## Section 6 — PyTorch Setup: Transforms, Dataset, Model

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Channels      : 3 (grayscale → RGB)")
print(f"✓ Normalisation : ImageNet mean/std [Zech et al. 2018]")
print(f"✓ Train augment : RandomHorizontalFlip + RandomRotation(10°)")
print(f"✓ Val augment   : none (consistent evaluation)")

✓ Image size    : 224×224
✓ Channels      : 3 (grayscale → RGB)
✓ Normalisation : ImageNet mean/std [Zech et al. 2018]
✓ Train augment : RandomHorizontalFlip + RandomRotation(10°)
✓ Val augment   : none (consistent evaluation)


In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ ChestXrayDataset defined


In [ ]:
test_ds    = ChestXrayDataset(clients['Hospital_A'], transform=val_transform)
image, lbl = test_ds[0]
print(f"Dataset size : {len(test_ds):,}")
print(f"Image shape  : {image.shape}")
print(f"Pixel range  : {image.min():.3f} – {image.max():.3f}")
print(f"Label        : {lbl.item()} ({'Pathology' if lbl.item()==1 else 'No Finding'})")
print("\n✓ Image loads correctly — pipeline ready")

Dataset size : 61,505
Image shape  : torch.Size([3, 224, 224])
Pixel range  : -2.118 – 2.413
Label        : 0.0 (No Finding)

✓ Image loads correctly — pipeline ready


In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]")
print(f"✓ Final layer : 512 → 1 (binary classification)")
print(f"✓ Loss        : BCEWithLogitsLoss")
print(f"✓ Optimiser   : Adam lr=1e-4")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]

✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]
✓ Final layer : 512 → 1 (binary classification)
✓ Loss        : BCEWithLogitsLoss
✓ Optimiser   : Adam lr=1e-4
✓ Parameters  : 11,177,025
✓ Rounds      : 10


In [ ]:
SAMPLE_FRACTION = 1.0
TRAIN_SPLIT     = 0.85
VAL_SPLIT       = 0.10
# remaining 0.05 = test

train_clients = {}
val_clients   = {}
test_clients  = {}

for name, data in clients.items():
    data_sample = data.sample(
        frac=SAMPLE_FRACTION, random_state=42
    ).reset_index(drop=True)

    n       = len(data_sample)
    n_train = int(n * TRAIN_SPLIT)
    n_val   = int(n * VAL_SPLIT)

    train_clients[name] = data_sample.iloc[:n_train].reset_index(drop=True)
    val_clients[name]   = data_sample.iloc[n_train:n_train+n_val].reset_index(drop=True)
    test_clients[name]  = data_sample.iloc[n_train+n_val:].reset_index(drop=True)

print(f"=== 10% Dataset | 85/10/5 Split ===\n")
print(f"{'Client':<14} {'Train':>8} {'Val':>6} {'Test':>6} {'Pathology%':>12}")
print("-" * 50)
for name in HOSPITAL_NAMES:
    print(f"{name:<14} {len(train_clients[name]):>8,} "
          f"{len(val_clients[name]):>6,} "
          f"{len(test_clients[name]):>6,} "
          f"{train_clients[name]['label'].mean()*100:>11.1f}%")
print(f"\n✓ train/val/test splits ready — 85/10/5")

=== 10% Dataset | 85/10/5 Split ===

Client            Train    Val   Test   Pathology%
--------------------------------------------------
Hospital_A       52,279  6,150  3,076        60.0%
Hospital_B        9,056  1,065    534        99.7%
Hospital_C       29,695  3,493  1,748         8.3%
Hospital_D        1,270    149     76        68.6%
Hospital_E        2,986    351    176         9.7%

✓ train/val/test splits ready — 85/10/5


## Section 7 — Evaluation Function

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics  : AUC + FNR (overall, per sex, per age group)")
print("  Citation : Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023")

✓ evaluate_client() defined
  Metrics  : AUC + FNR (overall, per sex, per age group)
  Citation : Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023


## Section 8 — Flower Base Client

In [ ]:
class HospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe      # training data
        self.val_dataframe = val_dataframe  # validation data
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy()
                for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
                len(self.val_dataframe), m

print("✓ HospitalClient defined")
print("  fit()      → trains on train_clients")
print("  evaluate() → validates on val_clients after each round")

✓ HospitalClient defined
  fit()      → trains on train_clients
  evaluate() → validates on val_clients after each round


## Section 9 — FedAvg (McMahan et al. 2017)
Algorithm 1: Server aggregates w_{t+1} = Σ_k (n_k/n) · w_k^{t+1}

In [ ]:
def make_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return HospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print(f"✓ client_fn factory defined")

✓ client_fn factory defined


In [ ]:
import logging
logging.getLogger('flwr').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

class FedAvgWithSave(fl.server.strategy.FedAvg):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.final_parameters = None

    def aggregate_fit(self, server_round, results, failures):
        aggregated = super().aggregate_fit(server_round, results, failures)
        if aggregated[0] is not None:
            self.final_parameters = aggregated[0]
        return aggregated

    def aggregate_evaluate(self, server_round, results, failures):
        aggregated = super().aggregate_evaluate(server_round, results, failures)

        if results:
            print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
            for _, res in results:
                name = res.metrics.get('client_name', '?')
                auc  = res.metrics.get('auc', float('nan'))
                fnr  = res.metrics.get('fnr', float('nan'))
                print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
            aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
            valid_aucs = [a for a in aucs if not (a != a)]
            if valid_aucs:
                print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                      f"Variance : {float(np.var(valid_aucs)):.6f}")

        return aggregated

fedavg_strategy = FedAvgWithSave(
    fraction_fit          = 1.0,
    fraction_evaluate     = 1.0,
    min_fit_clients       = NUM_CLIENTS,
    min_evaluate_clients  = NUM_CLIENTS,
    min_available_clients = NUM_CLIENTS,
    on_fit_config_fn      = lambda rnd: {'epochs': 3, 'lr': 1e-4}
)
print("✓ FedAvg strategy ready [McMahan et al. 2017]")
print(f"  Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"  Per-round validation output enabled")

✓ FedAvg strategy ready [McMahan et al. 2017]
  Rounds: 10 | Epochs/round: 3 | Clients: 5
  Per-round validation output enabled


In [ ]:
import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("FedAvg — McMahan et al. 2017")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 50)

t0 = time.time()

fedavg_history = fl.simulation.start_simulation(
    client_fn        = make_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = fedavg_strategy,
    client_resources = {'num_gpus': 1.0}
)

elapsed = time.time() - t0
print(f"\n{'='*50}")
print(f"✓ FedAvg complete in {elapsed/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in fedavg_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

FedAvg — McMahan et al. 2017
Rounds: 10 | Epochs/round: 3 | Clients: 5
Split: 85/10/5 | Batch: 512


2026-06-29 21:03:46,780	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=7270) 
(ClientAppActor pid=7270)             This is a deprecated feature. It will be removed
(ClientAppActor pid=7270)             entirely in future versions of Flower.
(ClientAppActor pid=7270)         
(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=7270) 
(ClientAppActor pid=7270)             This is a deprecated feature. It will be removed
(ClientAppActor pid=7270)             e


── Round 1/10 Validation ──
  Hospital_B     AUC: 0.7724  FNR: 0.3484
  Hospital_E     AUC: 0.7377  FNR: 0.3800
  Hospital_A     AUC: 0.7488  FNR: 0.3601
  Hospital_C     AUC: 0.7561  FNR: 0.3618
  Hospital_D     AUC: 0.7861  FNR: 0.3585
  Mean AUC : 0.7602 | Variance : 0.000295


(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=7270) 
(ClientAppActor pid=7270)             This is a deprecated feature. It will be removed
(ClientAppActor pid=7270)             entirely in future versions of Flower.
(ClientAppActor pid=7270)         
(ClientAppActor pid=7270) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=7270) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=7270)   self.pid = os.fork()
(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app impo


── Round 2/10 Validation ──
  Hospital_E     AUC: 0.7620  FNR: 0.4000
  Hospital_A     AUC: 0.7591  FNR: 0.4054
  Hospital_B     AUC: 0.6817  FNR: 0.4049
  Hospital_D     AUC: 0.7760  FNR: 0.3868
  Hospital_C     AUC: 0.7699  FNR: 0.3993
  Mean AUC : 0.7497 | Variance : 0.001193


...


── Round 8/10 Validation ──
  Hospital_A     AUC: 0.7299  FNR: 0.3700
  Hospital_B     AUC: 0.6394  FNR: 0.3522
  Hospital_E     AUC: 0.7068  FNR: 0.4200
  Hospital_D     AUC: 0.7742  FNR: 0.3774
  Hospital_C     AUC: 0.7108  FNR: 0.3823
  Mean AUC : 0.7122 | Variance : 0.001898


(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=7270) 
(ClientAppActor pid=7270)             This is a deprecated feature. It will be removed
(ClientAppActor pid=7270)             entirely in future versions of Flower.
(ClientAppActor pid=7270)         
(ClientAppActor pid=7270) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=7270) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=7270)   self.pid = os.fork()
(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app impo


── Round 9/10 Validation ──
  Hospital_E     AUC: 0.7047  FNR: 0.3600
  Hospital_A     AUC: 0.7118  FNR: 0.3310
  Hospital_B     AUC: 0.4893  FNR: 0.3004
  Hospital_C     AUC: 0.6948  FNR: 0.3345
  Hospital_D     AUC: 0.7413  FNR: 0.3208
  Mean AUC : 0.6684 | Variance : 0.008258


(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=7270) 
(ClientAppActor pid=7270)             This is a deprecated feature. It will be removed
(ClientAppActor pid=7270)             entirely in future versions of Flower.
(ClientAppActor pid=7270)         
(ClientAppActor pid=7270) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=7270) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=7270)   self.pid = os.fork()
(ClientAppActor pid=7270) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app impo


── Round 10/10 Validation ──
  Hospital_A     AUC: 0.7085  FNR: 0.3464
  Hospital_D     AUC: 0.7742  FNR: 0.3302
  Hospital_C     AUC: 0.6991  FNR: 0.3720
  Hospital_B     AUC: 0.5860  FNR: 0.3145
  Hospital_E     AUC: 0.7074  FNR: 0.3600
  Mean AUC : 0.6950 | Variance : 0.003701

✓ FedAvg complete in 305.6 minutes

Loss per round:
  Round  1 : 0.2465
  Round  2 : 0.2446
  Round  3 : 0.2395
  Round  4 : 0.2544
  Round  5 : 0.2772
  Round  6 : 0.2817
  Round  7 : 0.2854
  Round  8 : 0.2848
  Round  9 : 0.3145
  Round 10 : 0.3052


In [ ]:
import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

fedavg_metrics = {}
final_params   = parameters_to_ndarrays(fedavg_strategy.final_parameters)

for name, data in test_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in final_params]
    )))
    fedavg_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = fedavg_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_results = pd.DataFrame(rows).set_index('Client')
print("=== FedAvg Results — McMahan et al. 2017 ===\n")
print(df_results.to_string())

valid_aucs = [fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (fedavg_metrics[n]['auc'] != fedavg_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")

=== FedAvg Results — McMahan et al. 2017 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.6997  0.3332     65.34  0.6975  0.3311  0.7031  0.3358
Hospital_B  0.8931  0.2852     71.54     NaN  0.2888  0.8863  0.2796
Hospital_C  0.7039  0.3154     64.36  0.6845  0.3421  0.7310  0.2778
Hospital_D  0.7188  0.2679     68.42  0.7017  0.2353  0.7727  0.3182
Hospital_E  0.7262  0.3571     63.64  0.8019  0.2222  0.5747  0.6000

AUC Variance (valid hospitals only) : 0.005332
Valid hospitals : 5/5


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/fedavg_full_metrics.json', 'w') as f:
    json.dump(fedavg_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in final_params])),
    f'{SAVE_DIR}/fedavg_full_model.pth'
)

print("✓ FedAvg metrics saved → fedavg_full_metrics.json")
print("✓ FedAvg model saved  → fedavg_full_model.pth")

✓ FedAvg metrics saved → fedavg_full_metrics.json
✓ FedAvg model saved  → fedavg_full_model.pth
